#**WE_Chatbot_RAG_Explanation**

# WE Chatbot – RAG-based Customer Support Assistant

This notebook explains the design, architecture, and implementation
of a Retrieval-Augmented Generation (RAG) system built to provide
intelligent customer support for Telecom Egypt (WE).

The focus of this notebook is **explainability**, **system design**,
and **engineering decisions**


## Problem Statement

Telecom Egypt (WE) customer support information is distributed across
multiple web pages and documents.

Traditional chatbots often fail because they:
- Hallucinate answers
- Cannot cite official sources
- Struggle with Arabic-English bilingual support

To solve these issues, we adopt a **Retrieval-Augmented Generation (RAG)**
architecture that grounds responses in official WE content.


## Why Retrieval-Augmented Generation (RAG)?

RAG combines:
- **Information Retrieval** (Vector Search)
- **Text Generation** (Large Language Models)

Benefits:
- Factual accuracy
- Source grounding
- Reduced hallucination
- Better multilingual handling

This makes RAG suitable for enterprise-grade customer support systems.

## System Architecture

1. Data Collection (WE web pages & documents)
2. Text Cleaning & Chunking
3. Embedding Generation (Sentence Transformers)
4. Vector Storage (FAISS)
5. Query Retrieval
6. Answer Generation (LLM)
7. Streamlit User Interface

Imports & Configuration

In [ ]:
import os
import faiss
import pickle
import torch
import numpy as np

from typing import List, Tuple
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from docx import Document as DocxDocument
from PyPDF2 import PdfReader
from bs4 import BeautifulSoup
from PIL import Image
import pytesseract


## Design Decisions

- No LangChain or orchestration frameworks were used
- Full control over retrieval, filtering, and prompting
- Lightweight CPU-compatible models
- Production-oriented structure


Configuration Constants

In [ ]:
VECTOR_STORE_DIR = "data/vector_store"
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
LLM_MODEL = "google/mt5-base"

TOP_K = 6
MAX_CONTEXT_CHARS = 1800

device = torch.device("cpu")

Load Vector Store

In [ ]:
index = faiss.read_index(os.path.join(VECTOR_STORE_DIR, "we_faiss.index"))

with open(os.path.join(VECTOR_STORE_DIR, "documents.pkl"), "rb") as f:
    documents = pickle.load(f)

with open(os.path.join(VECTOR_STORE_DIR, "metadata.pkl"), "rb") as f:
    metadata = pickle.load(f)

Embedding & LLM Models

In [ ]:
embedder = SentenceTransformer(EMBEDDING_MODEL)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL).to(device)

Document Loading Functions

In [ ]:
def load_pdf(path):
    text = ""
    reader = PdfReader(path)
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text


def load_docx(path):
    doc = DocxDocument(path)
    return "\n".join([p.text for p in doc.paragraphs])


def load_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def load_html(path):
    with open(path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")
        return soup.get_text(separator="\n")


def load_image(path):
    img = Image.open(path)
    return pytesseract.image_to_string(img)

Unified Document Loader

In [ ]:
def load_user_document(path: str) -> str:
    ext = path.split(".")[-1].lower()

    if ext == "pdf":
        return load_pdf(path)
    elif ext == "docx":
        return load_docx(path)
    elif ext == "txt":
        return load_txt(path)
    elif ext == "html":
        return load_html(path)
    elif ext in ["jpg", "jpeg", "png"]:
        return load_image(path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")

Chunking Strategy

In [ ]:
def chunk_text(text, max_chars=MAX_CONTEXT_CHARS):
    chunks = []
    start = 0

    while start < len(text):
        end = min(len(text), start + max_chars)
        chunks.append(text[start:end])
        start = end

    return chunks


Chunking improves retrieval precision and prevents exceeding LLM context limits.

Adding User Documents

In [ ]:
def add_user_documents(paths: List[str]):
    global documents, metadata, index

    new_docs, new_meta = [], []

    for path in paths:
        text = load_user_document(path)
        chunks = chunk_text(text)

        for i, chunk in enumerate(chunks):
            new_docs.append(chunk)
            new_meta.append({
                "source": os.path.basename(path),
                "page": os.path.basename(path),
                "chunk_id": i + 1
            })

    embeddings = embedder.encode(new_docs)
    index.add(np.array(embeddings, dtype="float32"))

    documents.extend(new_docs)
    metadata.extend(new_meta)

Core RAG Function

In [ ]:
def ask_we_bot(question: str) -> Tuple[str, List[dict]]:
    query_vec = embedder.encode([question])
    distances, indices = index.search(query_vec, TOP_K)

    retrieved_docs, retrieved_meta = [], []
    seen_sources = set()

    keywords = [
        "WE Pay","subscribe","subscription","activate","wallet",
        "اشترك","تفعيل","باقة","فاتورة","محفظة"
    ]

    for i in indices[0]:
        text = documents[i]
        meta = metadata[i]

        if any(k.lower() in text.lower() for k in keywords):
            source_id = (meta["source"], meta["chunk_id"])
            if source_id not in seen_sources:
                retrieved_docs.append(text)
                retrieved_meta.append(meta)
                seen_sources.add(source_id)

    if not retrieved_docs:
        return "المعلومة غير متوفرة في موقع WE الرسمي.", []

    context = "\n\n".join(retrieved_docs[:TOP_K])


## Prompt Engineering Strategy

- Enforces language consistency
- Prevents hallucination
- Restricts answers to retrieved context
- Simulates a professional WE support agent


Answer Generation

In [ ]:
prompt = f"""
You are a professional customer support assistant for Telecom Egypt (WE).

Use ONLY the information in the context below.
Answer in the same language as the question.

CONTEXT:
{context}

QUESTION:
{question}
"""

inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)

outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer, retrieved_meta

Example Query

In [ ]:
answer, sources = ask_we_bot("ازاي اشترك في WE Pay؟")
print(answer)
sources

## Known Limitations

- CPU-only inference
- Precomputed embeddings
- Streamlit Cloud dependency installation issues

## Conclusion

This notebook demonstrates a complete, explainable,
and production-oriented RAG system suitable for
enterprise customer support scenarios.